In [ ]:
from langchain_core.tools import tool, InjectedToolArg
from langchain_cohere import ChatCohere
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from dotenv import load_dotenv
from typing import Annotated
import requests
import os
load_dotenv()

True

In [151]:
EXCHANGE_RATE_API_KEY  = os.getenv("EXCHANGE_RATE_API_KEY")

In [152]:
# Tool to get Latest Exchange Rates Scaling Factor

@tool
def get_exchange_rate_scaling_factor(from_currency: str, to_currency: str) -> float:
    """Takes two currency codes and returns the exchange rate scaling factor from the first currency to the second."""

    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/{from_currency}/{to_currency}"
    response = requests.get(url)
    conversion_rate = response.json()['conversion_rate']
    
    return conversion_rate

# Tool to get the total converted amount given the amount and the scaling factor

@tool
def convert_currency(amount: float,scaling_factor: Annotated[float, InjectedToolArg]) -> float:
    """Takes an amount and a scaling factor and returns the converted amount."""
    return amount * scaling_factor


In [153]:
# Tool Binding
# Binding tools help LLM to understand it has these tools which it can use. 

llm = ChatGroq(model="llama-3.3-70b-versatile")
llm_cohere = ChatCohere(model='command-a-03-2025')
llm_with_tool = llm.bind_tools([get_exchange_rate_scaling_factor, convert_currency])
llm_cohere_with_tool = llm.bind_tools([get_exchange_rate_scaling_factor, convert_currency])

In [154]:
# To Store Message history
messages = []

In [155]:
# User Query
query = "How much is 10 USD in INR?"

messages.append(HumanMessage(content=query))

In [156]:
# Tool Calling
# LLM recognize if its need to use the tool to answer the question or not

response = llm_with_tool.invoke(messages)
messages.append(response)
print(response)

#Since LLM recognize it needs to use the tool, in reponse it will show a tool_calls which will tell which tools the LLM think it needs to use
print(response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': '5pr7frpek', 'function': {'arguments': '{"from_currency":"USD","to_currency":"INR"}', 'name': 'get_exchange_rate_scaling_factor'}, 'type': 'function'}, {'id': 'qb87ge1j7', 'function': {'arguments': '{"amount":10}', 'name': 'convert_currency'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 331, 'total_tokens': 370, 'completion_time': 0.070914933, 'completion_tokens_details': None, 'prompt_time': 0.016138799, 'prompt_tokens_details': None, 'queue_time': 0.048185511, 'total_time': 0.087053732}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cf295-a791-7002-853a-d4d2d0010c4b-0' tool_calls=[{'name': 'get_exchange_rate_scaling_factor', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': '5pr7frpek', 'type': 'tool_call'}, {'name':

In [157]:
for tool_call in response.tool_calls:

    if tool_call['name'] == "get_exchange_rate_scaling_factor":
        sacling_factor_tool_message = get_exchange_rate_scaling_factor.invoke(tool_call)

        # Content always return string so we need to convert it to float before using it in the next tool call
        sacling_factor = float(sacling_factor_tool_message.content)
        messages.append(sacling_factor_tool_message)

    elif tool_call['name'] == "convert_currency":

        # Adding the scaling factor to the tool call args so that it can be used in the tool function
        tool_call['args']['scaling_factor'] = sacling_factor
        converted_amount_tool_message = convert_currency.invoke(tool_call)
        print(converted_amount_tool_message.content)
        messages.append(converted_amount_tool_message)

925.1139999999999


In [158]:
print(messages)

[HumanMessage(content='How much is 10 USD in INR?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5pr7frpek', 'function': {'arguments': '{"from_currency":"USD","to_currency":"INR"}', 'name': 'get_exchange_rate_scaling_factor'}, 'type': 'function'}, {'id': 'qb87ge1j7', 'function': {'arguments': '{"amount":10}', 'name': 'convert_currency'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 331, 'total_tokens': 370, 'completion_time': 0.070914933, 'completion_tokens_details': None, 'prompt_time': 0.016138799, 'prompt_tokens_details': None, 'queue_time': 0.048185511, 'total_time': 0.087053732}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf295-a791-7002-853a-d4d2d0010c4b-0', tool_calls=[{'name': 'get_exchange_rate_scaling_fac

In [179]:
response = llm_cohere_with_tool.invoke(messages)
print(response.content)

The exchange rate scaling factor from USD to INR is 82.55. 
So, 10 USD is approximately 825.5 INR.
